# 03 Character-level CNN (Model 2)

The CNN of Gregório et al. (2024), Figure 1: a character embedding, six convolutions with max-pooling, and a dense classifier.

It is run in two setups, like Model 1: **binary** (target DGA/legitimate) and **family-first** (target `legit` or a DGA family, collapsed with P(DGA) = 1 - P(legit)). The input is the full domain name as it is, TLD included (nothing is split off), and both setups use the sample weights. Metrics are computed on the DGA/legitimate decision, on the validation set; the test set is kept for the final results.

After loading and encoding the data, the notebook follows four steps:

1. **Define a predictor**: the CNN, which gives each domain a probability for every class.
2. **Define a loss**: weighted cross-entropy plus L1 and L2 penalties on the weights.
3. **Optimize the loss**: Adam on mini-batches, keeping the best epoch on validation.
4. **Test the model**: metrics on the DGA/legitimate decision.

The notebook defines each step and runs it; the code it uses is in `src/models/`:

- `charcnn.py`: the architecture of the CNN.
- `train.py`: the training loop (`fit`), which saves every epoch and continues an interrupted training.
- `tracking.py`: MLflow tracking of the runs, shared by every notebook.
- `plots.py`: the loss plot, redrawn after every epoch.
- `metrics.py`: the metrics, shared with the baselines.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

## Settings

Training follows the paper (Table 2): Adam, learning rate 1e-4, batch size 250, L1 1e-5 and L2 1e-4, 20 epochs, keeping the best epoch on validation (here by macro-F1). On a CPU one epoch over the full training set takes more than an hour, so use a GPU (e.g. Colab) for `SAMPLE = None`.

Each setup is saved after every epoch in its folder of `RUN_DIRS`, and tracked as one run of the MLflow experiment `MLFLOW_EXPERIMENT`.

In [ ]:
SAMPLE = 100_000   # None = full training set
EPOCHS = 20
LR = 1e-4          # Adam learning rate
BATCH_SIZE = 250
L1, L2 = 1e-5, 1e-4   # penalties on the convolution and dense weights
SEED = 0
EVAL_SET = "val"   # "test" only for the final results
WEIGHTED = True    # use the sample weights in the loss
CHECKPOINTS = ROOT / "checkpoints"   # trained models are saved here (not committed)
RUN = 1            # run number in the folder names below; a new number starts a new run
RUN_DIRS = {"binary": CHECKPOINTS / f"checkpoints_binary_cnn_{RUN}",
            "family": CHECKPOINTS / f"checkpoints_multifamily_cnn_{RUN}"}
MLFLOW_EXPERIMENT = "charcnn"   # stored in mlruns/ (not committed)

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from src.data.dataset import load_splits, sample_weights
from src.models import tracking
from src.models.charcnn import CharCNN
from src.models.encoding import class_codes, encode
from src.models.metrics import binary_metrics, recall_per_family
from src.models.plots import plot_losses
from src.models.train import (Split, fit, load_history, p_dga, penalty, pick_device,
                              predict_proba, set_seed)

device = pick_device()
print("device:", device, torch.cuda.get_device_name() if device.type == "cuda" else "")

train_full, val, test = load_splits()
train = train_full.sample(n=SAMPLE, random_state=SEED) if SAMPLE else train_full
evaluate = test if EVAL_SET == "test" else val
y_val = (val["label"] == "dga").astype(int).to_numpy()
y_eval = (evaluate["label"] == "dga").astype(int).to_numpy()
print(f"train: {len(train):,} rows (of {len(train_full):,}), "
      f"{EVAL_SET}: {len(evaluate):,} rows")

## Input encoding

The full domain name, TLD included, is the input. Each character becomes its ASCII code, padded with 0 at the end to 93 characters, the longest domain in the dataset, so no domain is cut (the paper used 70, the longest in its data). Classes are numbered from the full training set, so every family has an output even if a sample misses it.

In [ ]:
x_train, x_val, x_eval = (encode(part["domain"]) for part in (train, val, evaluate))
classes = {"label": sorted(train_full["label"].unique()),
           "family": sorted(train_full["family"].unique())}
print(f"input: {x_train.shape}, {len(classes['family'])} family classes")
print(train["domain"].iloc[0], "->", x_train[0][:20], "...")

## 1. Define a predictor

The predictor is the CNN of Figure 1. It maps an encoded domain to one score (logit) per class, and a softmax turns the scores into probabilities. The classes are `dga` and `legit` in the binary setup, and `legit` plus the DGA families in the family-first setup. In both, the DGA probability is P(DGA) = 1 - P(legit).

With a 93-character input the flattened size is 93 x 30 = 2790 (70 x 30 = 2100 in the paper).

In [ ]:
SETUPS = {"binary": "label", "family": "family"}   # setup -> target column


def make_predictor(task):
    """A new, untrained CNN with one output per class of `task`."""
    return CharCNN({task: len(classes[task])})


def predict_p_dga(model, x, task):
    """P(DGA) = 1 - P(legit), from the softmax over the classes of `task`."""
    return p_dga(predict_proba(model, x), task, classes)


model = make_predictor("family")
print(model)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

Before training, the weights are random: the predictor gives almost the same P(DGA) to every domain, so it cannot yet tell DGA from legitimate.

In [ ]:
examples = val.groupby("label").head(3)
model = make_predictor("label")
pd.DataFrame({"domain": examples["domain"].to_numpy(),
              "label": examples["label"].to_numpy(),
              "P(DGA), untrained": predict_p_dga(model, encode(examples["domain"]), "label").round(3)})

## 2. Define a loss

The loss of a batch of $B$ domains is the paper's loss (Table 2) with the sample weights added:

$$L = \frac{1}{B}\sum_{i=1}^{B} w_i \,\big(-\log P(y_i \mid x_i)\big) \;+\; \lambda_1 \sum |W| \;+\; \lambda_2 \sum W^2$$

- The first term is the cross-entropy: minus the log of the probability the predictor gives to the true class $y_i$ of domain $x_i$, weighted by its sample weight $w_i$. With `WEIGHTED = False`, every $w_i$ is 1.
- The other two terms are the L1 ($\lambda_1$ = 1e-5) and L2 ($\lambda_2$ = 1e-4) penalties on the convolution and dense weights $W$ (not the embedding, not the biases), added as Keras does (`penalty` in `src/models/train.py`).

In [ ]:
def loss_fn(model, logits, y, w, task):
    """Loss of one batch: weighted cross-entropy + L1/L2 penalty on the weights."""
    cross_entropy = F.cross_entropy(logits[task], y, reduction="none")   # -log P(y_i | x_i)
    return (cross_entropy * w).mean() + penalty(model, L1, L2)

The loss of the untrained binary predictor on the first batch of the training set:

In [ ]:
xb = torch.as_tensor(x_train[:BATCH_SIZE])
yb = torch.as_tensor(class_codes(train["label"].iloc[:BATCH_SIZE], classes["label"]))
wb = torch.as_tensor(np.array(train["weight"].iloc[:BATCH_SIZE]) if WEIGHTED
                     else np.ones(BATCH_SIZE), dtype=torch.float32)
with torch.no_grad():
    loss = loss_fn(model, model(xb), yb, wb, "label")
    reg = penalty(model, L1, L2)
print(f"loss {loss:.4f} = weighted cross-entropy {loss - reg:.4f} + penalty {reg:.4f}")

## 3. Optimize the loss

Adam (learning rate `LR`) minimizes the loss on shuffled mini-batches of `BATCH_SIZE` domains, for `EPOCHS` epochs. After every epoch, the predictor is scored on the validation set, and the weights of the epoch with the highest macro-F1 are kept (the paper keeps the best accuracy).

Two losses are recorded per epoch, both with `loss_fn`:

- $J_{train}$: the mean loss over the epoch's training batches (as Keras reports it), so it is measured while the weights are still changing.
- $J_{val}$: the loss on the whole validation set at the end of the epoch. The validation sample weights are computed in the same way as the training ones (`sample_weights`, from the validation family sizes), so the two losses are on the same scale.

`optimize` puts the three pieces together: the predictor (step 1), the loss (step 2) and the Adam optimizer. The training loop that runs them is `fit` in `src/models/train.py`.

In [ ]:
tracking.setup(MLFLOW_EXPERIMENT)   # the trainings below are logged to this MLflow experiment

w_train = train["weight"].to_numpy() if WEIGHTED else None
w_val = sample_weights(val) if WEIGHTED else None


def optimize(setup):
    """Train a new predictor for `setup`: Adam on loss_fn, run by fit (src/models/train.py)."""
    task = SETUPS[setup]
    set_seed(SEED)
    model = make_predictor(task)                              # step 1: the predictor
    loss = partial(loss_fn, task=task)                        # step 2: the loss
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)   # step 3: the optimizer
    history, p = fit(
        model, loss, optimizer,
        train=Split(x_train, class_codes(train[task], classes[task]), w_train),
        val=Split(x_val, class_codes(val[task], classes[task]), w_val, y_val),
        decide=task, classes=classes, folder=RUN_DIRS[setup],
        epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, device=device,
        params={"setup": setup, "task": task, "sample": SAMPLE, "lr": LR,
                "l1": L1, "l2": L2, "weighted": WEIGHTED})
    return model, history, p

One model per setup, each from the same seed. While a setup trains, its loss plot is redrawn after every epoch. After every epoch, `fit` saves the training in the setup's folder in `checkpoints/` (`RUN_DIRS`: `checkpoints_binary_cnn_1` and `checkpoints_multifamily_cnn_1` for `RUN = 1`):

- `epoch_01.pt`, `epoch_02.pt`, ...: the model weights after each epoch.
- `history.csv`: J_train, J_val and the validation metrics of every epoch so far.
- `last.pt`: everything needed to continue (weights, Adam state, shuffling state, history).
- `best.pt`: the weights of the kept epoch, written when the setup has finished.

Each setup is also an MLflow run (experiment `charcnn`, named after its folder): the settings, then J_train, J_val and the validation metrics of every epoch, and at the end the kept epoch, `history.csv`, the loss plot and the kept model. To browse the runs, run `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db` from the repository root and open http://127.0.0.1:5000.

If training stops (interrupted, crash, restart), run the notebook again from the top: each setup continues from its last saved epoch, in the same MLflow run and with the same shuffling as an uninterrupted run, and a setup that has finished is only reloaded. The settings are saved with the run, and a run cannot be continued with different settings: change `RUN` to train from scratch.

In [ ]:
models, histories, p_val = {}, {}, {}
for setup, task in SETUPS.items():
    print(f"--- {setup} ({len(classes[task])} classes), saved in {RUN_DIRS[setup]}")
    models[setup], histories[setup], p_val[setup] = optimize(setup)

In [ ]:
# The histories are read from the saved history.csv files, so this cell also plots a
# training that was interrupted (the setups and epochs saved so far).
histories = {setup: load_history(folder) for setup, folder in RUN_DIRS.items()}
histories = {setup: history for setup, history in histories.items() if history is not None}

# One loss panel per setup (the binary and family losses are on different scales), then
# validation macro-F1.
fig, axes = plt.subplots(1, len(histories) + 1, figsize=(15, 3.6))
for ax, (setup, history) in zip(axes, histories.items()):
    plot_losses(ax, history, f"Loss, {setup}")

ax = axes[-1]
for setup, history in histories.items():
    ax.plot(history.index, history["macro_f1"], marker="o", label=setup)
ax.set_xlabel("epoch")
ax.set_ylabel("macro-F1")
ax.set_title("Validation macro-F1")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Test the model

Each trained predictor gives P(DGA) for every domain of the evaluation set (`EVAL_SET`: the validation set for now, the test set for the final results). A domain is called DGA when P(DGA) >= 0.5. The metrics are those of the baselines, on the DGA/legitimate decision.

In [ ]:
# On the validation set, P(DGA) of the kept epoch was already computed in step 3 (same
# weights, same numbers), so the predictor only needs to run again on the test set.
p_dga_eval = {setup: p_val[setup] if EVAL_SET == "val" else predict_p_dga(models[setup], x_eval, task)
              for setup, task in SETUPS.items()}
results = {("Model 2", setup): binary_metrics(y_eval, p) for setup, p in p_dga_eval.items()}

In [ ]:
table = pd.DataFrame(results).T.round(4)
table.index.names = ["model", "setup"]
table

### Recall per DGA family

Recall of each DGA family in the two setups, families ordered from largest to smallest.

In [ ]:
family_sizes = train_full.loc[train_full["label"] == "dga", "family"].value_counts()
recall = pd.DataFrame({
    setup: recall_per_family(evaluate["family"], y_eval, (p >= 0.5).astype(int))
    for setup, p in p_dga_eval.items()}).reindex(family_sizes.index)
recall.columns = ["binary", "family-first"]

fig, ax = plt.subplots(figsize=(12, 3.6))
x = np.arange(len(recall))
for setup, marker in [("binary", "o"), ("family-first", "D")]:
    ax.scatter(x, recall[setup], s=24, marker=marker, label=setup, zorder=3)
ax.set_xlim(-1, len(recall))
ax.set_ylim(-0.03, 1.05)
ax.set_xlabel(f"DGA family, largest to smallest ({len(recall)} families)")
ax.set_ylabel("recall")
ax.set_title("Recall per DGA family, binary vs. family-first")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

### Families the CNN misses most

The 10 DGA families with the lowest recall in the family-first setup.

In [ ]:
recall.sort_values("family-first").head(10).round(3)